# Features Globales

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from typing import List, Sequence

def _as_date(df, date_col: str, out_col: str = "dt"):
    # Asume date_col puede venir como timestamp/string
    return df.withColumn(out_col, F.to_date(F.col(date_col)))

def _safe_abs_amount_col(amount_col: str):
    return F.abs(F.col(amount_col)).cast("double")

def _floor_int_amount(amount_col: str):
    return F.floor(_safe_abs_amount_col(amount_col)).cast("long")

def _eps_lit(eps: float):
    return F.lit(float(eps))

def _build_quantile_bins(df, col: str, probs: Sequence[float], rel_error: float = 0.01):
    """
    Devuelve una lista ordenada de cortes (boundaries) calculados SOLO con baseline.
    Incluye -inf y +inf al construir el binning.
    """
    cuts = df.approxQuantile(col, list(probs), rel_error)
    cuts = sorted(set([c for c in cuts if c is not None]))
    return cuts

def _assign_bin(col_expr, cuts: List[float], bin_col_name: str = "bin"):
    """
    Asigna bins 0..N según cortes (cuts). Bins:
      bin=0  : (-inf, cuts[0]]
      bin=i  : (cuts[i-1], cuts[i]]  (1..N-1)
      bin=N  : (cuts[-1], +inf)
    """
    if not cuts:
        return F.lit(0).alias(bin_col_name)

    expr = F.when(col_expr <= F.lit(cuts[0]), F.lit(0))
    for i in range(1, len(cuts)):
        expr = expr.when((col_expr > F.lit(cuts[i-1])) & (col_expr <= F.lit(cuts[i])), F.lit(i))
    expr = expr.otherwise(F.lit(len(cuts)))
    return expr.alias(bin_col_name)


## Benford deviation (1er dígito) por segmento

Qué entrega: por segment_cols (y opcionalmente por día/mes si lo incluyes como segmento), te da:

- mad_benford: mean(|p_obs - p_exp|)
- chi2_benford: sum((p_obs - p_exp)^2 / p_exp)
- y la distribución observada por dígito.

In [ ]:
def benford_deviation_first_digit(
    df,
    amount_col: str,
    segment_cols: List[str],
    eps: float = 1e-12
):
    amt = _safe_abs_amount_col(amount_col)

    # extraer 1er dígito ignorando ceros y decimales
    # 1) abs -> string
    # 2) remover todo lo que no sea dígito
    # 3) remover ceros a la izquierda
    # 4) primer char
    s = F.regexp_replace(amt.cast("string"), r"[^0-9]", "")
    s = F.regexp_replace(s, r"^0+", "")
    first_digit = F.substring(s, 1, 1)

    base = (
        df
        .withColumn("_amt", amt)
        .withColumn("_fd", first_digit)
        .where((F.col("_amt") > 0) & (F.col("_fd").rlike("^[1-9]$")))
        .withColumn("_d", F.col("_fd").cast("int"))
    )

    # conteos observados
    g = base.groupBy(*segment_cols, "_d").agg(F.count("*").alias("n"))
    totals = base.groupBy(*segment_cols).agg(F.count("*").alias("n_total"))
    g = g.join(totals, on=segment_cols, how="inner").withColumn("p_obs", F.col("n") / F.col("n_total"))

    # p esperado Benford: log10(1 + 1/d)
    p_exp = (F.log10(F.lit(1.0) + F.lit(1.0) / F.col("_d"))).alias("p_exp")
    g = g.withColumn("p_exp", p_exp)

    # MAD y Chi-square por segmento
    stats = (
        g.groupBy(*segment_cols)
        .agg(
            F.avg(F.abs(F.col("p_obs") - F.col("p_exp"))).alias("mad_benford"),
            F.sum(((F.col("p_obs") - F.col("p_exp")) ** 2) / (F.col("p_exp") + _eps_lit(eps))).alias("chi2_benford"),
            F.max("n_total").alias("n_total")
        )
    )

    # Si quieres además la tabla por dígito (útil para visualizar)
    dist = g.select(*segment_cols, F.col("_d").alias("digit"), "n", "n_total", "p_obs", "p_exp")

    return stats, dist


In [ ]:
#Uso Típico por día y por tip_cta
df_day = _as_date(df_tranfs, "event_date", "dt")
ben_stats, ben_dist = benford_deviation_first_digit(
    df_day,
    amount_col="mto_trf",
    segment_cols=["dt", "tip_cta"]
)


## Digit-preference / “roundness” (0/00/000, entropía últimos 2 dígitos, modo)

In [ ]:
def digit_preference_roundness(
    df,
    amount_col: str,
    segment_cols: List[str],
    eps: float = 1e-12
):
    amt_int = _floor_int_amount(amount_col)

    base = (
        df
        .withColumn("_amt_int", amt_int)
        .where(F.col("_amt_int") > 0)
        .withColumn("_last1", F.pmod(F.col("_amt_int"), F.lit(10)))
        .withColumn("_last2", F.pmod(F.col("_amt_int"), F.lit(100)))
        .withColumn("_last3", F.pmod(F.col("_amt_int"), F.lit(1000)))
    )

    # % terminando en 0, 00, 000
    simple = (
        base.groupBy(*segment_cols)
        .agg(
            F.count("*").alias("n"),
            F.avg(F.when(F.col("_last1") == 0, 1.0).otherwise(0.0)).alias("pct_end_0"),
            F.avg(F.when(F.col("_last2") == 0, 1.0).otherwise(0.0)).alias("pct_end_00"),
            F.avg(F.when(F.col("_last3") == 0, 1.0).otherwise(0.0)).alias("pct_end_000"),
        )
    )

    # Entropía de últimos 2 dígitos (0..99): -sum(p log p)
    c_last2 = base.groupBy(*segment_cols, "_last2").agg(F.count("*").alias("cnt_last2"))
    w_seg = Window.partitionBy(*segment_cols)
    c_last2 = c_last2.withColumn("p", F.col("cnt_last2") / F.sum("cnt_last2").over(w_seg))
    ent_last2 = (
        c_last2.groupBy(*segment_cols)
        .agg(
            (-F.sum(F.col("p") * F.log(F.col("p") + _eps_lit(eps)))).alias("entropy_last2")
        )
    )

    # Modo y % montos repetidos (share del monto más frecuente)
    c_amt = base.groupBy(*segment_cols, "_amt_int").agg(F.count("*").alias("cnt_amt"))
    w_rank = Window.partitionBy(*segment_cols).orderBy(F.desc("cnt_amt"), F.desc("_amt_int"))
    mode = (
        c_amt.withColumn("rn", F.row_number().over(w_rank))
        .where(F.col("rn") == 1)
        .select(
            *segment_cols,
            F.col("_amt_int").alias("mode_amount_int"),
            F.col("cnt_amt").alias("mode_count")
        )
    )
    mode = mode.join(simple.select(*segment_cols, "n"), on=segment_cols, how="inner") \
               .withColumn("mode_share", F.col("mode_count") / F.col("n")) \
               .drop("n")

    # Merge final
    out = (
        simple
        .join(ent_last2, on=segment_cols, how="left")
        .join(mode, on=segment_cols, how="left")
    )

    return out


In [ ]:
# Ejemplo de uso
df_day = _as_date(df_tranfs, "event_date", "dt")
round_feats = digit_preference_roundness(
    df_day,
    amount_col="mto_trf",
    segment_cols=["dt", "bco_dst"]  # ejemplo: por banco destino y día
)


## Drift por segmento: PSI y KS (aprox por bins de cuantiles del baseline)

In [ ]:
#Función principal

def distribution_drift_psi_ks(
    df,
    value_col: str,
    segment_cols: List[str],
    period_col: str,          # columna que indica baseline/current
    baseline_value: str,
    current_value: str,
    n_bins: int = 10,         # deciles por defecto
    rel_error: float = 0.01,
    eps: float = 1e-6
):
    """
    Calcula PSI y KS (aprox por bins) por segmento entre baseline y current.
    - Bins definidos por cuantiles del baseline (global, no por segmento).
    - period_col debe tener baseline_value / current_value.
    """

    # Preparación (filtrar valores válidos)
    base = df.withColumn("_x", F.col(value_col).cast("double")).where(F.col("_x").isNotNull())

    # baseline para cortes
    df_baseline = base.where(F.col(period_col) == F.lit(baseline_value))
    # cuantiles: p=0.1..0.9 si n_bins=10
    probs = [i / n_bins for i in range(1, n_bins)]
    cuts = _build_quantile_bins(df_baseline, "_x", probs=probs, rel_error=rel_error)

    # asignar bins a todo (baseline+current)
    binned = base.withColumn("_bin", _assign_bin(F.col("_x"), cuts, "bin"))

    # conteos por segmento, periodo, bin
    counts = (
        binned.groupBy(*segment_cols, period_col, "_bin")
        .agg(F.count("*").alias("cnt"))
    )

    # total por segmento y periodo
    totals = (
        binned.groupBy(*segment_cols, period_col)
        .agg(F.count("*").alias("n"))
    )

    counts = counts.join(totals, on=segment_cols + [period_col], how="inner") \
                   .withColumn("p", F.col("cnt") / F.col("n")) \
                   .select(*segment_cols, period_col, "_bin", "p")

    # pivot a baseline/current
    pivoted = (
        counts.groupBy(*segment_cols, "_bin")
        .pivot(period_col, [baseline_value, current_value])
        .agg(F.first("p"))
        .na.fill(0.0)
        .withColumnRenamed(baseline_value, "p_base")
        .withColumnRenamed(current_value, "p_cur")
    )

    # PSI: sum( (p_cur - p_base) * ln( (p_cur+eps)/(p_base+eps) ) )
    psi_df = (
        pivoted.withColumn(
            "_psi_term",
            (F.col("p_cur") - F.col("p_base")) *
            F.log((F.col("p_cur") + _eps_lit(eps)) / (F.col("p_base") + _eps_lit(eps)))
        )
        .groupBy(*segment_cols)
        .agg(F.sum("_psi_term").alias("psi"))
    )

    # KS aprox por bins: max |CDF_cur - CDF_base|
    w_bins = Window.partitionBy(*segment_cols).orderBy(F.col("_bin").asc()).rowsBetween(Window.unboundedPreceding, 0)
    cdf = (
        pivoted
        .withColumn("cdf_base", F.sum("p_base").over(w_bins))
        .withColumn("cdf_cur", F.sum("p_cur").over(w_bins))
        .withColumn("cdf_diff", F.abs(F.col("cdf_cur") - F.col("cdf_base")))
    )
    ks_df = cdf.groupBy(*segment_cols).agg(F.max("cdf_diff").alias("ks_approx"))

    # combinar
    out = psi_df.join(ks_df, on=segment_cols, how="inner")
    return out, pivoted, cuts


In [ ]:
#Cómo preparar period_col (baseline/current)
#Ejemplo: comparar mes actual vs mes anterior por segmento tip_cta y bco_dst.
df_day = _as_date(df_tranfs, "event_date", "dt").withColumn("ym", F.date_format("dt", "yyyy-MM"))

# define baseline y current
baseline_ym = "2025-11"
current_ym  = "2025-12"

df_period = (
    df_day
    .withColumn(
        "period",
        F.when(F.col("ym") == baseline_ym, F.lit("baseline"))
         .when(F.col("ym") == current_ym,  F.lit("current"))
         .otherwise(F.lit(None))
    )
    .where(F.col("period").isNotNull())
)

drift, drift_bins, cuts = distribution_drift_psi_ks(
    df_period,
    value_col="mto_trf",
    segment_cols=["tip_cta", "bco_dst"],
    period_col="period",
    baseline_value="baseline",
    current_value="current",
    n_bins=10
)
